In [0]:
import requests, json
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

now = datetime.now(ZoneInfo("America/Sao_Paulo"))

base_path = "dbfs:/Volumes/projetos/crypto/crypto_volume"
folder = f"{base_path}/raw/json/{now.strftime('%Y-%m-%d')}"

# ✔ cria diretório corretamente no UC Volume
dbutils.fs.mkdirs(folder)

url = "https://api.coingecko.com/api/v3/coins/markets"
params = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 100,
    "page": 1,
    "sparkline": False,
    "price_change_percentage": "1h,24h,7d"
}

r = requests.get(url, params=params, timeout=30)
r.raise_for_status()

file_path = f"{folder}/{now.strftime('%H-%M-%S')}.json"

payload = {
    "extracted_at": now.isoformat(),
    "data": r.json()
}

# ✔ grava no Volume corretamente
dbutils.fs.put(file_path, json.dumps(payload), overwrite=True)

print(f"✅ {len(r.json())} coins gravados em {file_path}")

In [0]:
from pyspark.sql import functions as F

VOLUME_PATH   = "/Volumes/projetos/crypto/crypto_volume"
CHECKPOINT    = f"{VOLUME_PATH}/checkpoints/bronze"
TABELA_BRONZE = "projetos.crypto.bronze_market_data"

(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{VOLUME_PATH}/schemas/bronze")
    .option("cloudFiles.inferColumnTypes", True)
    .load(f"{VOLUME_PATH}/raw/json/")
    .withColumn("_file_path", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(TABELA_BRONZE)
)

print("✅ Bronze carregado via Auto Loader")